# **Importing libraries**

In [99]:
import warnings
warnings.filterwarnings("ignore")

import gc
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score, roc_curve, auc

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageFile
from tqdm import tqdm
import re

In [100]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


# **Frame Extraction from Celeb-DF Videos**

In [101]:
CELEB_BASE = Path('/kaggle/input/datasets/reubensuju/celeb-df-v2')
FRAME_DIR = Path('/kaggle/working/celeb_frames')
FRAMES_PER_VIDEO = 10

def extract_frames(video_path, out_dir, n_frames=10):
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []
    indices = np.linspace(0, total - 1, n_frames, dtype=int)
    saved = []
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
        out_path = out_dir / f"frame_{i:03d}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved.append(str(out_path))
    cap.release()
    return saved

test_list_path = CELEB_BASE / 'List_of_testing_videos.txt'
if test_list_path.exists():
    with open(test_list_path) as f:
        test_videos = set(line.strip().split(' ')[1] for line in f if line.strip())
    print(f"Official test videos: {len(test_videos)}")
else:
    print("No test list found")
    test_videos = set()

label_map = {
    'Celeb-real':'real',
    'YouTube-real':'real',
    'Celeb-synthesis':'fake'
}

records = []
for folder, label in label_map.items():
    video_dir = CELEB_BASE / folder
    if not video_dir.exists():
        print(f"Missing folder: {video_dir}")
        continue
    for video_path in sorted(video_dir.glob('*.mp4')):
        rel   = f"{folder}/{video_path.name}"
        split = 'testing' if rel in test_videos else 'trainval'
        records.append({
            'video_path': str(video_path),
            'label':      label,
            'split':      split,
            'video_id':   f"{folder}__{video_path.stem}"
        })

video_df = pd.DataFrame(records)
print(video_df['split'].value_counts())
print(video_df['label'].value_counts())

Official test videos: 518
split
trainval    6011
testing      518
Name: count, dtype: int64
label
fake    5639
real     890
Name: count, dtype: int64


#  **Extract Frames to Disk**

In [102]:
file_names, labels, splits, video_ids = [], [], [], []

for _, row in tqdm(video_df.iterrows(), total=len(video_df), desc="Extracting frames"):
    out_dir = FRAME_DIR / row['split'] / row['label'] / row['video_id']
    frames  = extract_frames(Path(row['video_path']), out_dir, FRAMES_PER_VIDEO)
    for f in frames:
        file_names.append(f)
        labels.append(row['label'])
        splits.append(row['split'])
        video_ids.append(row['video_id'])

df = pd.DataFrame({'file_name': file_names, 'label': labels,
                   'split': splits, 'video_id': video_ids})

print(f"Total frames: {len(df)}")
print(df['label'].value_counts())

Extracting frames: 100%|██████████| 6529/6529 [10:52<00:00, 10.01it/s]

Total frames: 65281
label
fake    56390
real     8891
Name: count, dtype: int64


# **Train/Val/Test Split**

In [103]:
trainval_df = df[df['split'] == 'trainval']
test_df     = df[df['split'] == 'testing'].reset_index(drop=True)

unique_vids = trainval_df['video_id'].unique()
train_vids, val_vids = train_test_split(unique_vids, test_size=0.15, random_state=42)

train_df = trainval_df[trainval_df['video_id'].isin(train_vids)].reset_index(drop=True)
val_df   = trainval_df[trainval_df['video_id'].isin(val_vids)].reset_index(drop=True)

print(f"Overlap train∩val:  {len(set(train_df['video_id']) & set(val_df['video_id']))}")
print(f"Overlap train∩test: {len(set(train_df['video_id']) & set(test_df['video_id']))}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Val labels:",  val_df['label'].value_counts().to_dict())
print("Test labels:", test_df['label'].value_counts().to_dict())

Overlap train∩val:  0
Overlap train∩test: 0
Train: 51090 | Val: 9011 | Test: 5180
Val labels: {'fake': 7710, 'real': 1301}
Test labels: {'fake': 3400, 'real': 1780}


# **Encode labels**

In [104]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])
train_df['label_enc'] = le.transform(train_df['label'])
val_df['label_enc']   = le.transform(val_df['label'])
test_df['label_enc']  = le.transform(test_df['label'])
print("Classes:", le.classes_)

Classes: ['fake' 'real']


# **Dataset & Dataloaders**

In [105]:
IMG_SIZE = 224

class DeepfakeDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img   = Image.open(self.df.loc[idx, 'file_name']).convert('RGB')
        label = self.df.loc[idx, 'label_enc']
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.03),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.08))
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = DeepfakeDataset(train_df, train_transform)
val_dataset   = DeepfakeDataset(val_df,   val_transform)
test_dataset  = DeepfakeDataset(test_df,  val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")

Train: 1597 batches | Val: 282 | Test: 162


# **Model Defination**

In [106]:
class DeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        feat_dim = backbone.fc.in_features  
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = DeepfakeDetector(num_classes=2).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 24,563,266


# **Loss, Optimizer & Scheduler**

In [107]:
counts = np.array([train_df['label_enc'].value_counts()[i] for i in range(2)])
weights = torch.tensor(1.0 / counts, dtype=torch.float)
weights = (weights / weights.sum()).to(device)
print("Class weights:", weights)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 2e-5},
    {'params': model.head.parameters(),     'lr': 2e-4},
], weight_decay=1e-4)

EPOCHS    = 10
SAVE_PATH = '/kaggle/working/best_image_model.pth'

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr = [2e-5, 2e-4],
    steps_per_epoch = len(train_loader),
    epochs = EPOCHS,
    pct_start = 0.15,
    anneal_strategy = 'cos'
)

Class weights: tensor([0.1137, 0.8863], device='cuda:0')


# **Training Loop**

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs     = model(images)
            loss        = criterion(outputs, labels)
            total_loss += loss.item()
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

best_val_acc  = 0
patience      = 5
no_improve    = 0

for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler, device)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        no_improve   = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  Saved (val_acc={vl_acc:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

print(f"\nBest Val Accuracy: {best_val_acc:.4f}")

Epoch 01/10 | Train Loss: 0.7694 Acc: 0.5735 | Val Loss: 0.7376 Acc: 0.7129
  Saved (val_acc=0.7129)
Epoch 02/10 | Train Loss: 0.6181 Acc: 0.7385 | Val Loss: 0.6481 Acc: 0.8363
  Saved (val_acc=0.8363)
Epoch 03/10 | Train Loss: 0.5194 Acc: 0.8578 | Val Loss: 0.5968 Acc: 0.8965
  Saved (val_acc=0.8965)
Epoch 04/10 | Train Loss: 0.4774 Acc: 0.8976 | Val Loss: 0.6186 Acc: 0.8806
Epoch 05/10 | Train Loss: 0.4489 Acc: 0.9202 | Val Loss: 0.5773 Acc: 0.9147
  Saved (val_acc=0.9147)
Epoch 06/10 | Train Loss: 0.4362 Acc: 0.9352 | Val Loss: 0.5664 Acc: 0.9271
  Saved (val_acc=0.9271)


In [ ]:
IMAGE_SAVE_PATH = '/kaggle/working/best_image_model.pth'

model.load_state_dict(torch.load(IMAGE_SAVE_PATH))
model.eval()
print("Best image model loaded")

In [ ]:
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        outputs = model(images.to(device))
        probs   = torch.softmax(outputs, dim=1)[:, 1]
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

y_true  = np.array(all_labels)
y_pred  = np.array(all_preds)
y_probs = np.array(all_probs)

print("Inference complete")

# **Classification Report**

In [ ]:
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 Score : {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_true, y_probs):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=le.classes_))

# **Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix — Image Detection')
plt.tight_layout()
plt.savefig('/kaggle/working/image_confusion_matrix.png', dpi=120)
plt.show()

# **ROC Curve**

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='Blue', lw=2, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Image Detection')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/image_roc_curve.png', dpi=120)
plt.show()

# **Confidence Distribution**

In [ ]:
plt.figure(figsize=(6, 5))
plt.hist(y_probs[y_true == 0], bins=40, alpha=0.6, color='red',  label='Deepfake')
plt.hist(y_probs[y_true == 1], bins=40, alpha=0.6, color='blue', label='Real')
plt.xlabel('Predicted Probability (Real)')
plt.ylabel('Count')
plt.title('Confidence Distribution — Image Detection')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/image_confidence_dist.png', dpi=120)
plt.show()

# **Check Wrong Predictions**

In [ ]:
wrong_idx = np.where(y_pred != y_true)[0]
print(f"Total misclassified: {len(wrong_idx)} / {len(y_true)}")

n_show = min(10, len(wrong_idx))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, idx in enumerate(wrong_idx[:n_show]):
    img_path = test_df.iloc[idx]['file_name']
    img      = Image.open(img_path).convert('RGB').resize((224, 224))
    axes[i].imshow(img)
    true_label = le.classes_[y_true[idx]]
    pred_label = le.classes_[y_pred[idx]]
    conf       = y_probs[idx] if y_pred[idx] == 1 else 1 - y_probs[idx]
    axes[i].set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2f})", fontsize=9)
    axes[i].axis('off')

for j in range(n_show, len(axes)):
    axes[j].axis('off')

plt.suptitle('Misclassified Samples — Image Detection', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/image_errors.png', dpi=120)
plt.show()

# **Threshold**

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores  = [f1_score(y_true, (y_probs >= t).astype(int), average='macro') for t in thresholds]
best_t     = thresholds[np.argmax(f1_scores)]
best_f1    = max(f1_scores)

print(f"Default threshold (0.50) F1 : {f1_score(y_true, (y_probs >= 0.50).astype(int), average='macro'):.4f}")
print(f"Optimal threshold ({best_t:.2f}) F1 : {best_f1:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(thresholds, f1_scores, color='Blue', lw=2)
plt.axvline(best_t, color='blue', linestyle='--', label=f'Best threshold = {best_t:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Macro F1 Score')
plt.title('Threshold vs F1 Score — Image Detection')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/image_threshold_tuning.png', dpi=120)
plt.show()